<a href="https://colab.research.google.com/github/Ash100/Alignment/blob/main/Phylogenetic_Analysis_Protein_Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 **Bioinformatics Pipeline for Protein Sequence Alignment and Phylogenetics in Google Colab**

This notebook guides you through a complete pipeline for protein sequence phylogenetics:
- Uploading protein sequences
- Performing a multiple sequence alignment with MUSCLE
- Inferring a phylogenetic tree with IQ-TREE
- Exploring evolutionary models for proteins
- Visualizing the distance matrix

This pipeline is generated by **Dr. Ashfaq Ahmad**, you can watch complete video tutorial of this pipeline on [Bioinformatic Insights](https://youtu.be/IPc6yucfpnY).



### **Important Citations**

**MUSCLE:** Edgar RC. MUSCLE: multiple sequence alignment with high accuracy and high throughput. Nucleic Acids Res. 2004 Mar 19;32(5):1792-7. doi: 10.1093/nar/gkh340. PMID: 15034147; PMCID: PMC390337.

**IQ-Tree:** Bui Quang Minh, Heiko A. Schmidt, Olga Chernomor, Dominik Schrempf,
Michael D. Woodhams, Arndt von Haeseler, and Robert Lanfear (2020)
IQ-TREE 2: New models and efficient methods for phylogenetic inference
in the genomic era. Mol. Biol. Evol., in press.
https://doi.org/10.1093/molbev/msaa015

**Model Finder:** Subha Kalyaanamoorthy, Bui Quang Minh, Thomas KF Wong, Arndt von Haeseler,
and Lars S Jermiin (2017) ModelFinder: Fast model selection for
accurate phylogenetic estimates. Nature Methods, 14:587–589.
https://doi.org/10.1038/nmeth.4285

In [ ]:
# ========================
#@title 📦 Step 0: Install Required Tools
# ========================
!apt-get update -qq
!apt-get install -y muscle
!wget https://github.com/iqtree/iqtree2/releases/download/v2.2.6/iqtree-2.2.6-Linux.tar.gz
!tar -xzf iqtree-2.2.6-Linux.tar.gz
!mv iqtree-2.2.6-Linux/iqtree2 /usr/local/bin/iqtree2
!chmod +x /content/iqtree-2.2.6-Linux/bin/iqtree2


In [ ]:
# ========================
#@title 📁 Step 1: Upload FASTA file
# ========================
from google.colab import files
uploaded = files.upload()

# Get uploaded filename
fasta_file = list(uploaded.keys())[0]
print(f"Uploaded file: {fasta_file}")


In [ ]:
# ========================
#@title 🧬 Step 2: Align sequences with MUSCLE
# ========================
# Set filenames manually (for testing)
!cp $fasta_file input.fasta
!muscle -in /content/0.fas -out aligned.fasta

In [ ]:
# ========================
#@title 🌲 Step 3: Run IQ-TREE for Phylogeny Inference
# ========================
# IQ-TREE will automatically select the best-fit model using ModelFinder
!/content/iqtree-2.2.6-Linux/bin/iqtree2 -s /content/0.fasta -m MFP -bb 1000 -nt AUTO

# Output files will include:
# - *.treefile: the best tree
# - *.log: log of the process
# - *.iqtree: model details and statistics


# 🔍 Protein Model Selection in IQ-TREE

IQ-TREE uses **ModelFinder** to determine the best model of amino acid substitution. Here are some common protein models available:

- **LG**: General-purpose model derived from large alignments
- **WAG**: Based on globular protein alignments
- **JTT**: Based on protein families
- **Dayhoff**: Early protein substitution model
- **VT**: From vertebrate mitochondrial proteins
- **MtArt, MtRev**: Mitochondrial-specific models
- **HIVb, HIVw**: Based on HIV protein alignments

Additional modifiers:
- `+G`: Gamma-distributed rate variation
- `+I`: Proportion of invariable sites
- `+F`: Empirical amino acid frequencies
- `+R`: FreeRate model for heterogeneity

**Example model:** `LG+F+I+G4`

For the full list, run:
```bash
iqtree2 -m TESTONLY -s yourfile.fasta

**We will use `MFP`** that will enable IQ-Tree to launch a search on our dataset, and come up with the best results and model information.

In [ ]:
# ========================
#@title 📊 Step 4: Analyze Distance Matrix
# ========================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Read the .mldist file from IQ-TREE output
dist_file = "/content/0.mldist"

# Parse the file
with open(dist_file) as f:
    lines = f.readlines()

# First line: number of taxa (skip it)
matrix = []
taxa = []
for line in lines[1:]:
    parts = line.strip().split()
    taxa.append(parts[0])  # First column is taxon name
    matrix.append([float(x) for x in parts[1:]])

# Create a DataFrame
df = pd.DataFrame(matrix, index=taxa, columns=taxa)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(df, annot=False, cmap="viridis", xticklabels=True, yticklabels=True)
plt.title("Pairwise Evolutionary Distances (IQ-TREE)")
plt.tight_layout()

# Save the figure
plt.savefig("distance_heatmap.png", dpi=600)
print("Heatmap saved as 'distance_heatmap.png' (600 DPI)")

# Show the plot
plt.show()

---

### 🙌 Like this notebook?

If you found this helpful, please consider subscribing to **Bioinformatics Insights** for more content on tools, and tutorials!

👉 [Subscribe to Bioinformatics Insights](https://youtube.com/@bioinformaticsinsights?si=8SCaRpnoycm2oqND)

---
